# Testing `fairscape_models.sql`

In [1]:
#%pip install pydantic sqlalchemy duckdb-sqlalchemy

## Setup 

In [2]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [3]:
db_file = "integration_test.db"

In [4]:
try:
	os.remove(db_file)
except:
	pass

In [5]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

from fairscape_models.utils import readCrate
from fairscape_models.sql.models import *
from fairscape_models.sql.ingest import ROCrateIngestRequest
import sqlalchemy as sa
import pathlib

## Load ROCrate Tests

In [39]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [7]:
len(test_rocrate.metadataGraph)

20552

In [8]:
# TODO create a flamegraph of loading rocrate
# py-spy
# pip install py-spy
# py-spy record -o profile.svg -- python myscript.py 
#
# flameprof for cProfile stats
# python -m cProfile -o script.prof myscript.py
# 

In [9]:
# create an engine
engine = sa.create_engine(f"sqlite:///{db_file}")
# duckdb engine with file
#engine = sa.create_engine(f"duckdb:///{db_file}")
# duckdb engine in memory
#engine = sa.create_engine("duckdb:///:memory:")



# create table 
Base.metadata.create_all(engine)



In [10]:
type(engine)

sqlalchemy.engine.base.Engine

In [11]:
# create a session
session = sa.orm.Session(engine) 

#ingestRequest = ROCrateIngestRequest(
#	model=test_rocrate,
#	session=session,
#)

In [40]:
from fairscape_models.sql.conversion.construct import (
	ConvertROCrateToSQL, 
	ConvertDatasetToSQL,
	ConvertSoftwareToSQL,
	ConvertComputationToSQL
)

In [41]:
test_crate_metadata_instance = test_rocrate.getCrateMetadata()
test_computation_instance = test_rocrate.getComputations()[0]
test_software_instance = test_rocrate.getSoftware()[0]
test_dataset_instance = test_rocrate.getDatasets()[0]

In [46]:
output_copy = test_crate_metadata_instance.model_copy()
output_copy.hasPart = []


output = output_copy.model_dump_json(indent=2, by_alias=True)

# write file 
with open("test.json", "w") as jsonfile:
	jsonfile.write(output)

In [47]:
from fairscape_models.rocrate import ROCrateMetadataElem

In [49]:
with open("test.json", "r") as jsonfile:
	reading_elem = ROCrateMetadataElem.model_validate_json(jsonfile.read())

In [14]:
test_crate_metadata_instance.model_dump()['author']

'Leah V. Schaffer, Mengzhou Hu, Gege Qian, Kyung-Mee Moon, Abantika Pal, Neelesh Soni, Andrew P. Latham, Laura Pontano Vaites, Dorothy Tsai, Nicole M. Mattson, Katherine Licon, Robin Bachelder, Anthony Cesnik, Ishan Gaur, Trang Le, William Leineweber, Aji Palar, Ernst Pulido, Yue Qin, Xiaoyu Zhao, Christopher Churas, Joanna Lenkiewicz, Jing Chen, Keiichiro Ono, Dexter Pratt, Peter Zage, Ignacia Echeverria, Andrej Sali, J. Wade Harper, Steven P. Gygi, Leonard J. Foster, Edward L. Huttlin, Emma Lundberg & Trey Ideker'

In [15]:
crate_sql = ConvertROCrateToSQL(test_crate_metadata_instance)
computation_sql = ConvertComputationToSQL(test_computation_instance)
software_sql = ConvertSoftwareToSQL(test_software_instance)
dataset_sql = ConvertDatasetToSQL(test_dataset_instance)


In [16]:
dataset_sql_metadata = dataset_sql.__dict__.copy()
del dataset_sql_metadata['_sa_instance_state']

In [17]:
session.execute(sa.insert(DatasetSQL), dataset_sql_metadata)

In [18]:
session.scalar(
	sa.select(DatasetSQL)
)

In [19]:
computation_sql.author

[{'name': 'Leah V. Schaffer'}]

In [20]:
test_computation_instance.runBy

'Leah V. Schaffer'

In [21]:
# optimizing ingest for full ROCrate


In [22]:
from fairscape_models.rocrate import ROCrateMetadataFileElem, ROCrateV1_2
from fairscape_models.sql.utils import DetermineMetadataTypeSQL

def generateMetadata(inputCrate: ROCrateV1_2):	
	for crate_elem in inputCrate.metadataGraph:
		if isinstance(crate_elem, ROCrateMetadataFileElem):
			continue
		
		elemSQLType = DetermineMetadataTypeSQL(crate_elem.metadataType)
		match elemSQLType:
			case MetadataTypeEnumSQL.COMPUTATION:
				yield ConvertComputationToSQL(crate_elem)
			case MetadataTypeEnumSQL.SCHEMA:
				pass
			case MetadataTypeEnumSQL.ROCRATE:
				yield ConvertROCrateToSQL(crate_elem)
			case MetadataTypeEnumSQL.DATASET:
				yield ConvertDatasetToSQL(crate_elem)
			case MetadataTypeEnumSQL.SOFTWARE:
				yield ConvertSoftwareToSQL(crate_elem)


def generateIdentifiers(inputCrate: ROCrateV1_2):
	pass


def generateMembership(inputCrate: ROCrateV1_2):
	pass


def generateProv(inputCrate: ROCrateV1_2):
	pass

In [23]:
# insert just datasets 
test_rocrate.getDatasets()

[Dataset(guid='ark:59853/dataset-image-gene-node-attributes-with-locations', metadataType=['prov:Entity', 'https://w3id.org/EVI#Dataset'], name='Image Gene Node Attributes with Locations', isPartOf=[], author=[{'name': 'Robin Bachelder'}, {'name': 'Gege Qian'}, {'name': 'Neelesh Soni'}, {'name': 'Emma Lundberg '}, {'name': 'Anthony Cesnik'}, {'name': 'Steven P. Gygi'}, {'name': 'J. Wade Harper'}, {'name': 'Yue Qin'}, {'name': 'Joanna Lenkiewicz'}, {'name': 'Dorothy Tsai'}, {'name': 'Christopher Churas'}, {'name': 'Trey Ideker'}, {'name': 'William Leineweber'}, {'name': 'Abantika Pal'}, {'name': 'Ignacia Echeverria'}, {'name': 'Ernst Pulido'}, {'name': 'Andrew P. Latham'}, {'name': 'Andrej Sali'}, {'name': 'Peter Zage'}, {'name': 'Laura Pontano Vaites'}, {'name': 'Aji Palar'}, {'name': 'Jing Chen'}, {'name': 'Xiaoyu Zhao'}, {'name': 'Leonard J. Foster'}, {'name': 'Keiichiro Ono'}, {'name': 'Leah V. Schaffer'}, {'name': 'Edward L. Huttlin'}, {'name': 'Mengzhou Hu'}, {'name': 'Katherine L

In [24]:
for elem in generateMetadata(test_rocrate):
	session.add(elem)

In [25]:
session.flush()

In [26]:
metadataGen = generateMetadata(test_rocrate)

In [27]:
session.add(crate_sql)

In [28]:
session.flush()

In [31]:
session.scalars(sa.select(ROCrateMetadataElemSQL)).all()

In [30]:
# test digest authors
author_data = ingestRequest._digest_authors()



NameError: name 'ingestRequest' is not defined

In [ ]:
# check authors
author_data, existing_author_ids = ingestRequest._check_authors(author_data)

# if no id's are found author id is an empty dictionary
existing_author_ids

{}

In [ ]:
# write the authors
author_ids = ingestRequest._write_authors(author_data)

# join existing author ids to author_ids
full_author_ids = author_ids | existing_author_ids

In [ ]:
full_author_ids

{'Trey Ideker': 1,
 'Dorothy Tsai': 2,
 'Peter Zage': 3,
 'Emma Lundberg ': 4,
 'Jing Chen': 5,
 'Neelesh Soni': 6,
 'Xiaoyu Zhao': 7,
 'Trang Le': 8,
 'Aji Palar': 9,
 'J. Wade Harper': 10,
 'Nicole M. Mattson': 11,
 'Ignacia Echeverria': 12,
 'Yue Qin': 13,
 'Edward L. Huttlin': 14,
 'Abantika Pal': 15,
 'Ernst Pulido': 16,
 'Dexter Pratt': 17,
 'Mengzhou Hu': 18,
 'Joanna Lenkiewicz': 19,
 'Steven P. Gygi': 20,
 'Leonard J. Foster': 21,
 'Leah V. Schaffer': 22,
 'Andrej Sali': 23,
 'Katherine Licon': 24,
 'Anthony Cesnik': 25,
 'William Leineweber': 26,
 'Ishan Gaur': 27,
 'Kyung-Mee Moon': 28,
 'Keiichiro Ono': 29,
 'Andrew P. Latham': 30,
 'Robin Bachelder': 31,
 'Christopher Churas': 32,
 'Laura Pontano Vaites': 33,
 'Gege Qian': 34}

In [ ]:
author_ids

{'Trey Ideker': 1,
 'Dorothy Tsai': 2,
 'Peter Zage': 3,
 'Emma Lundberg ': 4,
 'Jing Chen': 5,
 'Neelesh Soni': 6,
 'Xiaoyu Zhao': 7,
 'Trang Le': 8,
 'Aji Palar': 9,
 'J. Wade Harper': 10,
 'Nicole M. Mattson': 11,
 'Ignacia Echeverria': 12,
 'Yue Qin': 13,
 'Edward L. Huttlin': 14,
 'Abantika Pal': 15,
 'Ernst Pulido': 16,
 'Dexter Pratt': 17,
 'Mengzhou Hu': 18,
 'Joanna Lenkiewicz': 19,
 'Steven P. Gygi': 20,
 'Leonard J. Foster': 21,
 'Leah V. Schaffer': 22,
 'Andrej Sali': 23,
 'Katherine Licon': 24,
 'Anthony Cesnik': 25,
 'William Leineweber': 26,
 'Ishan Gaur': 27,
 'Kyung-Mee Moon': 28,
 'Keiichiro Ono': 29,
 'Andrew P. Latham': 30,
 'Robin Bachelder': 31,
 'Christopher Churas': 32,
 'Laura Pontano Vaites': 33,
 'Gege Qian': 34}

In [ ]:
# TODO slow?
# ingestRequest._write_identifier_authors(full_author_ids)

In [ ]:
# check that author table has correct author information
session.scalar(sa.select(sa.func.count(AuthorSQL.id)))

34

In [ ]:
# check linked authors
session.scalar(sa.select(sa.func.count(AuthorIdentifierSQL.id)))

698633

In [ ]:
# digest identifiers
rocrate_identifiers = ingestRequest._digest_identifiers()
ingestRequest._write_identifiers(rocrate_identifiers)

session.flush()
session.commit()

In [ ]:
# check the identifiers

In [ ]:
# digest all crate elements 
crate_elements = ingestRequest._digest_iterate_elements()

In [ ]:
# write rocrate elements
ingestRequest._write_elements()

In [ ]:
session.commit()

In [ ]:
session.close()

## Test Query

In [ ]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [ ]:
from fairscape_models.sql.query import QueryByGUID, QueryResponse, SERIALIZE_TYPE
from fairscape_models.utils import readCrate
import sqlalchemy as sa
import pathlib

from fairscape_models.dataset import Dataset
from fairscape_models.software import Software
from fairscape_models.computation import Computation
from fairscape_models.rocrate import ROCrateMetadataElem

# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")

# create table 
# Base.metadata.create_all(engine)


In [ ]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [ ]:
test_crate_metadata = test_rocrate.getCrateMetadata()
test_crate_software = test_rocrate.getSoftware()[0]
test_crate_computation = test_rocrate.getComputations()[0]
test_crate_dataset = test_rocrate.getDatasets()[0]

In [ ]:
test_crate_dataset.fileFormat

'.tsv'

In [ ]:
session = sa.orm.Session(engine)

In [ ]:
def runQuery(GUID: str, session):
	query = QueryByGUID(GUID)
	results = query.execute(session)
	
	return results.transform()


In [ ]:
dataset_result = runQuery(test_crate_dataset.guid, session)

#dataset_query = QueryByGUID(test_crate_dataset.guid)
#dataset_query_response = dataset_query.execute(session)
#dataset_query_response.transform()

In [ ]:
computation_result = runQuery(test_crate_computation.guid, session)

In [ ]:
computation_query = QueryByGUID(test_crate_computation.guid)
computation_query_response = computation_query.execute(session)

In [ ]:
computation_query_response.rootEntity

In [ ]:
computation_query_response._transform_root_entity()

In [ ]:
computation_query_response._convert_metadata()

In [ ]:
computation_query_response.metadata

{'dateCreated': '2023-08-01',
 'name': 'Image Download from Human Protein Atlas',
 'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 'usedSoftware': [{'@id': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader'}],
 'datePublished': None,
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [ ]:
from fairscape_models.sql.conversion.construct import ConvertComputationToSQL


test_comp = ConvertComputationToSQL(test_crate_computation)

In [ ]:
computation_query_response.metadata

{'dateCreated': '2023-08-01',
 'name': 'Image Download from Human Protein Atlas',
 'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 'usedSoftware': [{'@id': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader'}],
 'datePublished': None,
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [ ]:
software_result = runQuery(test_crate_software.guid, session)

In [ ]:
crate_result = runQuery(test_crate_metadata.guid, session)

In [ ]:
test_crate_computation.metadataType

['prov:Activity', 'https://w3id.org/EVI#Computation']

In [ ]:
type(computation_result)

fairscape_models.computation.Computation

## Full ROCrate Query

In [ ]:
test_rocrate_guid = test_crate_metadata.guid

# query identifier metadata
#type_query = sa.select(IdentifiersSQL.metadataType).where(IdentifiersSQL.guid == test_rocrate_guid)

# membership query
children_guid_query = sa.select(MembershipSQL.childGUID, MembershipSQL.childType).where(MembershipSQL.parentGUID == test_rocrate_guid)

children_guid_results = session.execute(children_guid_query).all()

In [ ]:
test_rocrate_guid

'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'

In [ ]:
children_guid_results

[('ark:59853/software-cellmaps-imagedownloader', <MetadataTypeEnumSQL.SOFTWARE: 'SOFTWARE'>),
 ('ark:59853/dataset-image-gene-node-attributes-with-locations', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-gene-node-attributes', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/computation-image-download', <MetadataTypeEnumSQL.COMPUTATION: 'COMPUTATION'>),
 ('ark:59853/dataset-image-1174_c5_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1913_h4_2', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1521_a3_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-778_h1_4', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1193_d6_5', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1235_g2_2', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-418_h11_1', <MetadataTypeEnumSQL.DATASET: 'DATASET'>),
 ('ark:59853/dataset-image-1191_d7_1', 

In [ ]:
element_guid, element_type = children_guid_results[0]

In [ ]:
from fairscape_models.sql.query import TYPE_LOOKUP, METADATA_TYPE_PROPERTY

metadata = {}
root_entity_query = sa.select(TYPE_LOOKUP[element_type]).filter_by(guid=element_guid)
root_entity_results = session.scalars(root_entity_query).all()

# keywords
keyword_query = sa.select(KeywordSQL.keywordValue).filter_by(guid=element_guid)

# authors
author_identifier_query = sa.select(AuthorSQL.name, AuthorSQL.orcid).join(AuthorIdentifierSQL, AuthorIdentifierSQL.author_id == AuthorSQL.id).where(AuthorIdentifierSQL.identifier_guid==element_guid)

author_results = session.execute(author_identifier_query).all()

# hasPart
has_part_query = sa.select(MembershipSQL.childGUID).where(MembershipSQL.parentGUID==element_guid)
has_part_results = session.execute(has_part_query).all()

# isPartOf
is_part_of_query = sa.select(MembershipSQL.parentGUID).where(MembershipSQL.childGUID==element_guid)
is_part_of_results = session.execute(is_part_of_query).all()

match element_type:
	case MetadataTypeEnumSQL.DATASET:
		# if its a dataset usedBy/generatedBy
		used_by_query = sa.select(ComputationUsedDatasetSQL.computationGUID).where(ComputationUsedDatasetSQL.datasetGUID == element_guid)
		used_by_results = session.execute(used_by_query).all()


		generated_by_query = sa.select(ComputationGeneratedDatasetSQL.computationGUID).where(ComputationUsedDatasetSQL.datasetGUID == element_guid)
		generated_by_results = session.execute(generated_by_query).all()

		pass
	case MetadataTypeEnumSQL.SOFTWARE:
		# if its a software usedBy
		used_by_query = sa.select(ComputationSQL.guid).where(ComputationSQL.usedSoftware == element_guid)
		used_by_results = session.execute(used_by_query).all()

		pass
	case MetadataTypeEnumSQL.COMPUTATION:
	# if its a computation used/generated
		used_query = sa.select(ComputationUsedDatasetSQL.datasetGUID).where(ComputationUsedDatasetSQL.computationGUID == element_guid)
		used_results = session.execute(used_query).all()

		generated_query = sa.select(ComputationGeneratedDatasetSQL.datasetGUID).where(ComputationGeneratedDatasetSQL.computationGUID == element_guid)
		generated_results = session.execute(generated_query).all()



In [ ]:
metadata

{}